# 🤖 Advanced Machine Learning & Reinforcement Learning Project
### Course: Advanced Machine Learning (ELE 365)
**Faculty of Engineering – New Ismailia National University (NINU)**  
**Author**: Ammar Abdelfattah & Team  
**Focus**: Advanced Supervised Classification, Tabular Q-Learning (FrozenLake), and Deep Q-Networks (DQN)

---

## 📌 Project Overview
This project synthesizes advanced machine learning concepts studied throughout the course and practical sheets:
1. **Part I: Customer Churn Prediction (`telecom_churn.csv`)**:
   - Advanced feature engineering, class imbalance treatment, multi-model benchmarking (Regularized Logistic Regression, Random Forest, Gradient Boosting, MLP).
   - Precision-Recall trade-offs, ROC-AUC analysis, and feature importance attribution.
2. **Part II: Tabular Reinforcement Learning (FrozenLake-v2 - Sheet 3)**:
   - Markov Decision Process (MDP) formulation: States, Actions, Transitions, and Discounted Rewards.
   - Bellman Optimality update:
     $$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ r + \gamma \max_{a'} Q(s', a') - Q(s, a) \right]$$
   - $\epsilon$-greedy exploration decay and optimal policy convergence.
3. **Part III: Deep Q-Networks (DQN - Sheet 4)**:
   - Neural network value function approximation $Q(s, a; \theta)$.
   - Experience Replay Memory buffer and Target Network $\theta^-$ for temporal-difference stability.


In [ ]:
# 1. Environment Setup and Library Imports
import os
import sys
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score, roc_curve, precision_recall_curve
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier

print("Libraries imported successfully.")
np.random.seed(42)


---
## 📊 Part I: Advanced Classification & Churn Prediction
Predicting customer churn using `telecom_churn.csv`.


In [ ]:
# 2. Dataset Loading & Exploratory Data Analysis (EDA)
DATA_PATH = "telecom_churn.csv"

if os.path.exists(DATA_PATH):
    df = pd.read_csv(DATA_PATH)
    print(f"Loaded dataset with {df.shape[0]} rows and {df.shape[1]} columns.")
else:
    # Generate representative dataset if running standalone
    print("Generating representative telecom churn dataset for immediate verification...")
    n_samples = 3000
    df = pd.DataFrame({
        "account_length": np.random.randint(1, 240, n_samples),
        "international_plan": np.random.choice(["yes", "no"], n_samples, p=[0.1, 0.9]),
        "voice_mail_plan": np.random.choice(["yes", "no"], n_samples, p=[0.25, 0.75]),
        "number_vmail_messages": np.random.randint(0, 50, n_samples),
        "total_day_minutes": np.random.normal(180, 50, n_samples).clip(0, 350),
        "total_day_calls": np.random.randint(50, 150, n_samples),
        "total_eve_minutes": np.random.normal(200, 45, n_samples).clip(0, 350),
        "total_night_minutes": np.random.normal(200, 45, n_samples).clip(0, 350),
        "total_intl_minutes": np.random.normal(10, 3, n_samples).clip(0, 20),
        "customer_service_calls": np.random.poisson(1.5, n_samples),
        "churn": np.random.choice([0, 1], n_samples, p=[0.85, 0.15])
    })

print(df.head())
print("
Class Distribution:
", df.iloc[:, -1].value_counts(normalize=True))


In [ ]:
# 3. Data Preprocessing Pipeline
target_col = df.columns[-1]
X = df.drop(columns=[target_col])
y = df[target_col]

# Convert boolean/string target to binary integer
if y.dtype == object or y.dtype == bool:
    y = LabelEncoder().fit_transform(y)

num_cols = X.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X.select_dtypes(include=['object', 'category', 'bool']).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False), cat_cols)
    ]
)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
print(f"Train size: {X_train.shape}, Test size: {X_test.shape}")


In [ ]:
# 4. Multi-Model Benchmark Suite
models = {
    "Logistic Regression (L2)": LogisticRegression(max_iter=1000, class_weight='balanced'),
    "Random Forest": RandomForestClassifier(n_estimators=150, max_depth=10, class_weight='balanced', random_state=42),
    "Hist Gradient Boosting": HistGradientBoostingClassifier(max_iter=150, random_state=42),
    "Multi-Layer Perceptron (MLP)": MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=300, random_state=42)
}

benchmark_results = {}

plt.figure(figsize=(10, 6))

for name, clf in models.items():
    pipe = Pipeline([('prep', preprocessor), ('clf', clf)])
    pipe.fit(X_train, y_train)
    
    y_pred = pipe.predict(X_test)
    y_proba = pipe.predict_proba(X_test)[:, 1] if hasattr(pipe, "predict_proba") else y_pred
    
    auc = roc_auc_score(y_test, y_proba)
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    plt.plot(fpr, tpr, label=f"{name} (AUC = {auc:.3f})")
    
    benchmark_results[name] = {
        "ROC-AUC": round(auc, 4),
        "Pipeline": pipe,
        "Report": classification_report(y_test, y_pred, output_dict=True)
    }

plt.plot([0, 1], [0, 1], 'k--', alpha=0.7)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curves Comparison - Customer Churn Benchmark")
plt.legend(loc="lower right")
plt.grid(True, linestyle="--", alpha=0.5)
plt.show()

# Display Performance Summary Table
summary_df = pd.DataFrame([
    {
        "Model": name,
        "ROC-AUC": res["ROC-AUC"],
        "Precision (Churn)": round(res["Report"]["1"]["precision"], 3),
        "Recall (Churn)": round(res["Report"]["1"]["recall"], 3),
        "F1-Score (Churn)": round(res["Report"]["1"]["f1-score"], 3),
    }
    for name, res in benchmark_results.items()
])
print("
" + "="*60)
print("         MODEL PERFORMANCE COMPARISON")
print("="*60)
print(summary_df.to_string(index=False))


---
## 🧊 Part II: Tabular Reinforcement Learning - Q-Learning (FrozenLake-v2)
As assigned in **Sheet 3**:
Design a reinforcement learning agent to navigate a slippery grid environment:
- **States**: 16 grid cells ($4 \times 4$)
- **Actions**: 4 directional moves (0: Left, 1: Down, 2: Right, 3: Up)
- **Goal**: Reach state 15 (Goal, Reward = 1) without falling into Holes (H, Reward = 0).

### Bellman Optimality Update:
$$Q(s, a) \leftarrow Q(s, a) + \alpha \left[ r + \gamma \max_{a'} Q(s', a') - Q(s, a) \right]$$


In [ ]:
# 5. Environment Simulator & Q-Learning Agent
class FrozenLakeSimulator:
    """Simulates 4x4 FrozenLake gridworld with slippery stochastic transitions"""
    def __init__(self, is_slippery=True):
        self.nrow = 4
        self.ncol = 4
        self.n_states = 16
        self.n_actions = 4
        self.is_slippery = is_slippery
        # Holes at indices: 5, 7, 11, 12; Goal at 15
        self.holes = {5, 7, 11, 12}
        self.goal = 15
        self.reset()
        
    def reset(self):
        self.state = 0
        return self.state
        
    def step(self, action):
        if self.state in self.holes or self.state == self.goal:
            return self.state, 0, True, {}
            
        # Slippery transition dynamics
        if self.is_slippery:
            # 1/3 probability of intended action, 2/3 of perpendicular drifts
            actions = [action, (action - 1) % 4, (action + 1) % 4]
            chosen_action = np.random.choice(actions)
        else:
            chosen_action = action
            
        r, c = self.state // self.ncol, self.state % self.ncol
        if chosen_action == 0: c = max(0, c - 1)  # Left
        elif chosen_action == 1: r = min(self.nrow - 1, r + 1)  # Down
        elif chosen_action == 2: c = min(self.ncol - 1, c + 1)  # Right
        elif chosen_action == 3: r = max(0, r - 1)  # Up
        
        next_state = r * self.ncol + c
        self.state = next_state
        
        done = (next_state in self.holes) or (next_state == self.goal)
        reward = 1.0 if next_state == self.goal else 0.0
        return next_state, reward, done, {}

def train_q_learning(episodes=3000, alpha=0.1, gamma=0.99, epsilon_start=1.0, epsilon_decay=0.998, epsilon_min=0.01):
    env = FrozenLakeSimulator(is_slippery=True)
    Q = np.zeros((env.n_states, env.n_actions))
    
    epsilon = epsilon_start
    rewards_history = []
    
    for ep in range(episodes):
        state = env.reset()
        total_r = 0
        done = False
        
        while not done:
            # Epsilon-greedy action selection
            if np.random.rand() < epsilon:
                action = np.random.randint(env.n_actions)
            else:
                action = np.argmax(Q[state])
                
            next_state, reward, done, _ = env.step(action)
            
            # Q-learning Bellman update
            best_next_q = np.max(Q[next_state])
            td_target = reward + gamma * best_next_q * (not done)
            Q[state, action] += alpha * (td_target - Q[state, action])
            
            state = next_state
            total_r += reward
            
        epsilon = max(epsilon_min, epsilon * epsilon_decay)
        rewards_history.append(total_r)
        
    return Q, rewards_history

print("Training Q-Learning agent on FrozenLake...")
Q_table, rewards = train_q_learning(episodes=4000)

# Plot smoothed learning curve
window = 100
smoothed_rewards = pd.Series(rewards).rolling(window).mean()

plt.figure(figsize=(10, 4))
plt.plot(smoothed_rewards, color='darkgreen', linewidth=2)
plt.title(f"Q-Learning Convergence on FrozenLake (Rolling Window = {window})")
plt.xlabel("Episode")
plt.ylabel("Success Rate / Average Reward")
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()

# Display Learned Q-Table
print("
Final Learned Q-Table (16 States x 4 Actions [L, D, R, U]):")
print(np.round(Q_table, 3))


---
## 🧠 Part III: Deep Reinforcement Learning - Deep Q-Network (DQN)
As assigned in **Sheet 4**:
Extending Q-learning to high-dimensional continuous or large state spaces using a Deep Neural Network to approximate $Q(s, a; \theta)$.

Key components implemented:
1. **Q-Network & Target Network** ($	heta, 	heta^-$)
2. **Experience Replay Buffer** to break data correlation
3. **Loss minimization** via Mean Squared Bellman Error:
   $$L(\theta) = \mathbb{E}_{(s, a, r, s', d) \sim D} \left[ \left( r + \gamma \max_{a'} Q(s', a'; \theta^-)(1 - d) - Q(s, a; \theta) \right)^2 \right]$$


In [ ]:
# 6. Deep Q-Network (DQN) Architecture & Training Loop
from collections import deque
import random

class ReplayBuffer:
    def __init__(self, capacity=5000):
        self.buffer = deque(maxlen=capacity)
    def push(self, state, action, reward, next_state, done):
        self.buffer.append((state, action, reward, next_state, done))
    def sample(self, batch_size):
        batch = random.sample(self.buffer, batch_size)
        s, a, r, ns, d = zip(*batch)
        return np.array(s), np.array(a), np.array(r, dtype=np.float32), np.array(ns), np.array(d, dtype=np.float32)
    def __len__(self):
        return len(self.buffer)

class SimpleLinearDQN:
    """Vectorized Q-network approximator with target network stability"""
    def __init__(self, state_dim=16, action_dim=4, lr=0.01):
        self.state_dim = state_dim
        self.action_dim = action_dim
        self.lr = lr
        # Weight matrix: (state_dim, action_dim)
        self.W = np.random.randn(state_dim, action_dim) * 0.05
        self.W_target = self.W.copy()
        
    def forward(self, state_onehot, use_target=False):
        W = self.W_target if use_target else self.W
        return np.dot(state_onehot, W)
        
    def train_step(self, states, actions, rewards, next_states, dones, gamma=0.99):
        # Target computation: r + gamma * max_a' Q_target(s', a')
        next_q = self.forward(next_states, use_target=True)
        max_next_q = np.max(next_q, axis=1)
        targets = rewards + gamma * max_next_q * (1.0 - dones)
        
        # Predictions
        current_q = self.forward(states, use_target=False)
        
        # Gradient update for selected actions
        batch_size = len(states)
        for i in range(batch_size):
            s_vec = states[i]
            a = actions[i]
            error = targets[i] - current_q[i, a]
            # Gradient of 0.5 * error^2 w.r.t W[:, a] is -error * s_vec
            self.W[:, a] += self.lr * error * s_vec

    def update_target_network(self):
        self.W_target = self.W.copy()

def state_to_onehot(state, dim=16):
    vec = np.zeros(dim)
    vec[state] = 1.0
    return vec

def train_dqn_agent(episodes=1500, batch_size=32, target_update_freq=50):
    env = FrozenLakeSimulator(is_slippery=False)  # Deterministic testbed for clear DQN validation
    dqn = SimpleLinearDQN(state_dim=16, action_dim=4, lr=0.05)
    memory = ReplayBuffer(capacity=2000)
    
    epsilon = 1.0
    epsilon_min = 0.05
    epsilon_decay = 0.995
    dqn_rewards = []
    
    for ep in range(episodes):
        state = env.reset()
        done = False
        ep_r = 0
        
        while not done:
            s_vec = state_to_onehot(state)
            if np.random.rand() < epsilon:
                action = np.random.randint(4)
            else:
                action = np.argmax(dqn.forward(s_vec))
                
            next_state, reward, done, _ = env.step(action)
            ns_vec = state_to_onehot(next_state)
            
            memory.push(s_vec, action, reward, ns_vec, done)
            state = next_state
            ep_r += reward
            
            if len(memory) >= batch_size:
                b_s, b_a, b_r, b_ns, b_d = memory.sample(batch_size)
                dqn.train_step(b_s, b_a, b_r, b_ns, b_d)
                
        if ep % target_update_freq == 0:
            dqn.update_target_network()
            
        epsilon = max(epsilon_min, epsilon * epsilon_decay)
        dqn_rewards.append(ep_r)
        
    return dqn, dqn_rewards

print("Training Deep Q-Network (DQN) agent...")
trained_dqn, dqn_history = train_dqn_agent(episodes=1200)

plt.figure(figsize=(10, 4))
plt.plot(pd.Series(dqn_history).rolling(50).mean(), color='crimson', linewidth=2)
plt.title("DQN Convergence Curve on Grid Environment")
plt.xlabel("Episode")
plt.ylabel("Reward Moving Average")
plt.grid(True, linestyle="--", alpha=0.6)
plt.show()


---
## 🎯 Key Takeaways & Discussion
1. **Supervised vs. Reinforcement Learning**: Supervised models require labeled target ground-truths, while RL agents discover optimal decision policies purely via evaluative environmental reward feedback.
2. **Q-Learning vs. DQN**: Tabular Q-learning is provably convergent for small finite state-action spaces, but fails as state dimensions scale. DQN scales to continuous and massive representations using neural function approximations and replay memory.
